# APUSH LEQ Grader SLM — v1 QLoRA train + base-vs-tuned eval (Colab)

Runs the Day-3 gate on a GPU: train the v1 QLoRA adapter, then eval it on both tracks and put the numbers on the board.

**Before running:** Runtime → Change runtime type → **GPU** (T4 is enough for a 0.5B model).

Order: check GPU → clone → install → (HF token) → train → eval litmus → eval real → print base-vs-tuned table.

`artifacts/data/` is committed to the repo, so no data regeneration is needed — the clone already has `train_chat.jsonl`, `eval_cases.jsonl`, and `eval_real_cases.jsonl`.

## 1. Confirm the GPU is attached

In [ ]:
!nvidia-smi
import torch
print('CUDA available:', torch.cuda.is_available())

assert torch.cuda.is_available(), 'No GPU. Runtime -> Change runtime type -> GPU, then rerun.'

Thu Jul  9 16:42:17 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Clone the repo

Public repo — the committed `artifacts/data/` comes with it. Re-run-safe: pulls latest if the folder already exists.

In [19]:
import os
REPO = 'https://github.com/aryanjverma/slm.git'
if not os.path.isdir('/content/slm'):
    !git clone $REPO /content/slm
else:
    !cd /content/slm && git pull
%cd /content/slm
!wc -l artifacts/data/train_chat.jsonl artifacts/data/eval_cases.jsonl artifacts/data/eval_real_cases.jsonl

Already up to date.
/content/slm
   1000 artifacts/data/train_chat.jsonl
    198 artifacts/data/eval_cases.jsonl
     72 artifacts/data/eval_real_cases.jsonl
   1270 total


## 3. Install training extras

Installs the package with the `[train]` extra (unsloth / trl / peft / bitsandbytes). Takes a few minutes; a pip resolver warning about pre-installed Colab packages is normal.

In [ ]:
!pip install -q -e ".[train]"

Updating 0c39132..0c43965
Fast-forward
 docs/v1_eval_results.md             |  12 ++
 notebooks/colab_train_eval_v1.ipynb | 242 ++++++++++++++++++++++++++++++++++--
 scripts/eval_hf_model.py            |  33 ++++-
 scripts/summarize_from_results.py   |  74 +++++++++++
 src/apush_frq_grader_slm/eval.py    |   6 +
 tests/test_core.py                  |  12 +-
 6 files changed, 363 insertions(+), 16 deletions(-)
 create mode 100644 docs/v1_eval_results.md
 create mode 100644 scripts/summarize_from_results.py


## 4. (Optional) Hugging Face token

Avoids rate-limited base-model downloads. Add a Colab secret named `HF_TOKEN` (🔑 in the left sidebar), or skip this cell to download anonymously.

In [4]:
import os
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print('HF_TOKEN set from Colab secrets.')
except Exception as e:
    print('No HF_TOKEN (downloading anonymously):', e)

No HF_TOKEN (downloading anonymously): Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.


## 5. Train the v1 QLoRA adapter

Defaults: `Qwen/Qwen2.5-0.5B-Instruct`, 300 steps, LoRA rank 16 — a few minutes on a T4. Writes the adapter to `artifacts/models/apush-frq-grader-v1/`.

In [ ]:
!cd /content/slm && python scripts/train_qlora.py \
  --model Qwen/Qwen2.5-0.5B-Instruct \
  --data artifacts/data/train_chat.jsonl \
  --output artifacts/models/apush-frq-grader-v1 \
  --max-steps 300

## 6. Refresh the deterministic baselines

Regenerates `inflated_prompted_base` and `apush_grader_reference` on the same 198-case litmus set so the tuned model is compared against them on identical inputs. CPU-fast.

In [ ]:
!cd /content/slm && python -m apush_frq_grader_slm.cli.run_eval \
  --eval-path artifacts/data/eval_cases.jsonl \
  --output-dir artifacts/eval

## 6b. Tokenizer deps for eval

The eval script loads the Qwen tokenizer in a fresh subprocess, which needs 
`sentencepiece`/`tiktoken` to build the fast tokenizer. These are now in the 
`[train]` extra, but install them explicitly here so an already-installed runtime is covered.

In [4]:
!pip install -q sentencepiece tiktoken

## 7. Eval the tuned model — litmus track (the gate)

Runs the tuned adapter over the synthetic contract + adversarial set. This is the base-vs-tuned regression signal.

In [5]:
!cd /content/slm && python scripts/eval_hf_model.py \
  --model artifacts/models/apush-frq-grader-v1 \
  --model-name apush_frq_grader_v1 \
  --eval-path artifacts/data/eval_cases.jsonl \
  --output-dir artifacts/eval

Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
Loading weights: 100% 290/290 [00:00<00:00, 446.79it/s]
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a 

## 8. Eval the tuned model — real track (College Board agreement)

Row-score agreement and QWK vs the ~72 real CB essays. Eval only — these never entered training.

In [8]:
!cd /content/slm && python scripts/eval_hf_model.py \
  --model artifacts/models/apush-frq-grader-v1 \
  --model-name apush_frq_grader_v1 \
  --eval-path artifacts/data/eval_real_cases.jsonl \
  --real-eval \
  --output-dir artifacts/eval

Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
Loading weights: 100% 290/290 [00:00<00:00, 620.43it/s]
/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/bnb.py:373: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
apush_frq_grader_v1:   0% 0/72 [00:00<?, ?case/s]The following generation flags are not valid a

## 8b. Recompute the CB summary from saved results (no re-generation)

If step 8 finished generating but errored while summarizing, the per-case results are already saved. This rebuilds `apush_frq_grader_v1_real_summary.jsonl` from them in seconds — no model re-run. Run it **before** re-running step 7, which would overwrite a legacy shared results file. Then run cell 9/9b to see and write the numbers.

In [12]:
import os
if os.path.isdir('/content/slm'):
    os.chdir('/content/slm')

# Recompute the CB (real) summary from the saved per-case results WITHOUT
# re-running generation. Use this if step 8 finished generating but failed while
# summarizing (e.g. the old QWK crash). Prefers the new *_real_results.jsonl and
# falls back to the legacy shared *_results.jsonl left by a pre-fix run.
candidates = [
    'artifacts/eval/apush_frq_grader_v1_real_results.jsonl',
    'artifacts/eval/apush_frq_grader_v1_results.jsonl',
]
results = next((p for p in candidates if os.path.exists(p)), candidates[0])
print('Recomputing CB summary from:', results)

!cd /content/slm && python scripts/summarize_from_results.py \
  --results {results} \
  --eval-path artifacts/data/eval_real_cases.jsonl \
  --model-name apush_frq_grader_v1 \
  --output-dir artifacts/eval \
  --real-eval

Recomputing CB summary from: artifacts/eval/apush_frq_grader_v1_results.jsonl
python3: can't open file '/content/slm/scripts/summarize_from_results.py': [Errno 2] No such file or directory


## 8c. Diagnose the CB json_valid failures (truncation vs malformed)

The real-track `json_valid` is low. This buckets the failures from the saved per-case results: **unparseable/truncated** (output ran past `--max-new-tokens` before closing the JSON → a quick fix is a higher token cap) vs **parsed-but-invalid** (wrong schema or out-of-range scores → a genuine generalization/data problem). Also prints the `notes` distribution and a few failing response tails.

In [20]:
import os, json
from collections import Counter

if os.path.isdir('/content/slm'):
    os.chdir('/content/slm')
from apush_frq_grader_slm.filters import parse_grade_json
from apush_frq_grader_slm.rubric import validate_grade_payload

# Diagnose WHY the CB (real) json_valid rate is low: truncated output (started
# JSON, no closing brace -> raise --max-new-tokens) vs. parsed-but-invalid grades
# (wrong schema / out-of-range scores -> a real generalization/data problem).
candidates = [
    'artifacts/eval/apush_frq_grader_v1_real_results.jsonl',
    'artifacts/eval/apush_frq_grader_v1_results.jsonl',
]
path = next((p for p in candidates if os.path.exists(p)), None)
assert path, 'No real results file found — run step 8 (or 8b) first.'
rows = [json.loads(l) for l in open(path) if l.strip()]
print(f"{len(rows)} results from {path}\n")

valid = sum(1 for r in rows if r.get('structured_output_valid'))
unparseable = truncated = parsed_but_invalid = 0
reasons = Counter()
notes = Counter(r.get('notes', '?') for r in rows)

for r in rows:
    resp = (r.get('response') or '').strip()
    payload, _ = parse_grade_json(resp)
    if payload is None:
        unparseable += 1
        if not resp.endswith('}'):          # opened JSON but never closed it
            truncated += 1
    else:
        ok, why = validate_grade_payload(payload)
        if not ok:
            parsed_but_invalid += 1
            for w in (why if isinstance(why, (list, tuple)) else [why]):
                reasons[str(w)] += 1

print(f"valid grades:        {valid}/{len(rows)}")
print(f"unparseable JSON:    {unparseable}  (no closing '}}': {truncated} -> likely truncation)")
print(f"parsed but invalid:  {parsed_but_invalid}")
print("\nnotes distribution:")
for k, v in notes.most_common():
    print(f"  {v:3d}  {k}")
if reasons:
    print("\nwhy parsed-but-invalid:")
    for k, v in reasons.most_common():
        print(f"  {v:3d}  {k}")
print("\nsample failing response tails (last 100 chars):")
shown = 0
for r in rows:
    if not r.get('structured_output_valid'):
        print("  ...", repr((r.get('response') or '')[-100:]))
        shown += 1
        if shown >= 4:
            break

72 results from artifacts/eval/apush_frq_grader_v1_results.jsonl

valid grades:        21/72
unparseable JSON:    11  (no closing '}': 9 -> likely truncation)
parsed but invalid:  40

notes distribution:
   51  invalid_or_missing_json
   15  ok
    4  hallucination_or_rewrite
    2  generic_ungrounded_feedback

why parsed-but-invalid:
   40  total_mismatch
   15  out_of_range_thesis
   15  out_of_range_contextualization
    5  out_of_range_evidence
    2  out_of_range_analysis_reasoning

sample failing response tails (last 100 chars):
  ... ' by examining the specific actions taken during the period, such as the annexation of Hawaii."\n  }\n}'
  ... ' the US government\'s actions during this period were influenced by changing foreign policies."\n  }\n}'
  ... 'tical ideologies like the Progressive movement. The essay includes several examples, such as the use'
  ... 'hey were part of a larger historical context, such as the Cold War and the rise of communism."\n  }\n}'


## 9. Base-vs-tuned summary table

Reads every summary written to `artifacts/eval/` and prints the litmus comparison. The gate is met when `apush_frq_grader_v1` beats `inflated_prompted_base` on grounding and robustness.

In [ ]:
import json, glob, os

# cwd can reset to /content after a runtime disconnect/reconnect; the %cd
# magic from cell 5 does not persist. Re-anchor before reading relative paths.
if os.path.isdir('/content/slm'):
    os.chdir('/content/slm')

def load(path):
    with open(path) as f:
        return [json.loads(l) for l in f if l.strip()]

# Litmus summaries: baselines live in summary.jsonl; tuned model in its own *_summary.jsonl.
rows = {}
sp = 'artifacts/eval/summary.jsonl'
if os.path.exists(sp):
    for r in load(sp):
        rows[r['model_name']] = r
for p in glob.glob('artifacts/eval/*_summary.jsonl'):
    if 'slice' in p or 'real' in p or p.endswith('summary.jsonl') and os.path.basename(p) == 'summary.jsonl':
        continue
    for r in load(p):
        rows[r['model_name']] = r

order = ['inflated_prompted_base', 'apush_frq_grader_v1', 'apush_grader_reference']
ranked = [rows[m] for m in order if m in rows] + [r for m, r in rows.items() if m not in order]

hdr = f"{'model':<28} {'n':>4} {'json':>6} {'rubric':>7} {'ground':>7} {'robust':>7} {'total':>6}"
print('LITMUS (198-case synthetic contract + adversarial)')
print(hdr); print('-' * len(hdr))
for r in ranked:
    print(f"{r['model_name']:<28} {r['count']:>4} {r['structured_output_valid_rate']:>6.2f} "
          f"{r['rubric_accuracy_mean']:>7.2f} {r['evidence_grounding_rate']:>7.2f} "
          f"{r['robustness_mean']:>7.2f} {r['total_score_mean']:>6.2f}")

# Real track (tuned model only).
rp = 'artifacts/eval/apush_frq_grader_v1_real_summary.jsonl'
if os.path.exists(rp):
    r = load(rp)[0]
    print('\nREAL (College Board essays)')
    print(f"  cases={r['count']}  json_valid={r['structured_output_valid_rate']:.2f}")
    print(f"  row exact={r['exact_match_rate']:.2f}  row within-1={r['within_one_rate']:.2f}")
    print(f"  total exact={r['total_exact_match_rate']:.2f}  total within-1={r['total_within_one_rate']:.2f}  QWK={r['qwk']}")

## 9b. Write the results table to a file

Rebuilds the litmus (and real, if present) tables from `artifacts/eval/*_summary.jsonl` and writes them to `artifacts/eval/v1_eval_results.md`. That path is gitignored, so it never blocks `git pull` (cell 5); the curated snapshot lives at `docs/v1_eval_results.md`. Prints the same table it writes. Run this after steps 7–9.

In [ ]:
import json, glob, os
from datetime import datetime

# cwd can reset to /content after a runtime disconnect; re-anchor before I/O.
if os.path.isdir('/content/slm'):
    os.chdir('/content/slm')

def load(path):
    with open(path) as f:
        return [json.loads(l) for l in f if l.strip()]

# Collect litmus summaries: baselines in summary.jsonl, tuned model in its own file.
rows = {}
sp = 'artifacts/eval/summary.jsonl'
if os.path.exists(sp):
    for r in load(sp):
        rows[r['model_name']] = r
for p in glob.glob('artifacts/eval/*_summary.jsonl'):
    base = os.path.basename(p)
    if 'slice' in base or 'real' in base or base == 'summary.jsonl':
        continue
    for r in load(p):
        rows[r['model_name']] = r

order = ['inflated_prompted_base', 'apush_frq_grader_v1', 'apush_grader_reference']
ranked = [rows[m] for m in order if m in rows] + [r for m, r in rows.items() if m not in order]

lines = [
    f"# v1 Eval Results — auto-generated {datetime.now():%Y-%m-%d %H:%M}",
    "",
    "> Regenerated by the notebook summary cell; numbers come straight from",
    "> `artifacts/eval/*_summary.jsonl`. Curated copy: `docs/v1_eval_results.md`.",
    "",
    "## Litmus (198-case synthetic contract + adversarial)",
    "",
    "| model | n | json | rubric | ground | robust | total |",
    "|-------|---|------|--------|--------|--------|-------|",
]
for r in ranked:
    lines.append(
        f"| {r['model_name']} | {r['count']} | {r['structured_output_valid_rate']:.2f} | "
        f"{r['rubric_accuracy_mean']:.2f} | {r['evidence_grounding_rate']:.2f} | "
        f"{r['robustness_mean']:.2f} | {r['total_score_mean']:.2f} |"
    )

rp = 'artifacts/eval/apush_frq_grader_v1_real_summary.jsonl'
if os.path.exists(rp):
    r = load(rp)[0]
    lines += [
        "",
        "## Real (College Board essays)",
        "",
        "| metric | value |",
        "|--------|-------|",
        f"| cases | {r['count']} |",
        f"| json_valid | {r['structured_output_valid_rate']:.2f} |",
        f"| row exact | {r['exact_match_rate']:.2f} |",
        f"| row within-1 | {r['within_one_rate']:.2f} |",
        f"| total exact | {r['total_exact_match_rate']:.2f} |",
        f"| total within-1 | {r['total_within_one_rate']:.2f} |",
        f"| QWK | {r['qwk']} |",
    ]

report = "\n".join(lines) + "\n"
out_path = 'artifacts/eval/v1_eval_results.md'
os.makedirs('artifacts/eval', exist_ok=True)
with open(out_path, 'w', encoding='utf-8') as f:
    f.write(report)

print(report)
print(f"[written to {os.path.abspath(out_path)}]")

## 10. (Optional) Save the adapter to Google Drive

The adapter is `.gitignore`d and Colab storage is ephemeral. Copy it to Drive to keep it after the runtime disconnects.

In [10]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/slm_models
!cp -r artifacts/models/apush-frq-grader-v1 /content/drive/MyDrive/slm_models/
!ls -la /content/drive/MyDrive/slm_models/apush-frq-grader-v1

ValueError: mount failed